## **Bibliotecas**

In [34]:
%pip install groq

from dotenv import load_dotenv
from groq import Groq
import os

Note: you may need to restart the kernel to use updated packages.


## **Configurações Iniciais**


Carrega as variáveis de ambiente do arquivo .env

In [35]:
load_dotenv()

True

Inicializa o cliente da Groq para fazer chamadas à API

In [36]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

### Quickstart do Groq

In [37]:
chat_completion = client.chat.completions.create(
    messages=[         
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are essential in today's technology-driven world, and their importance can be understood from several perspectives:

1. **Efficient Processing**: Fast language models enable rapid processing of large amounts of text data, which is crucial in various applications such as:
	* Sentiment analysis
	* Text classification
	* Named entity recognition
	* Machine translation
	* Question answering
2. **Real-time Response**: Fast language models facilitate real-time responses in applications like:
	* Chatbots and virtual assistants
	* Voice assistants (e.g., Siri, Alexa, Google Assistant)
	* Live language translation
	* Social media monitoring
3. **Scalability**: Fast language models can handle massive volumes of data, making them suitable for large-scale applications like:
	* Data analytics
	* Content moderation
	* Recommendation systems
4. **Improved User Experience**: Fast language models provide quick and accurate results, leading to:
	* Enhanced user satisfaction
	* Incre

## **Agente**

In [38]:
class Agent:
    def __init__(self, client: Groq, system):
        """
        Inicializa o agente, um prompt e uma lista de mensagens para manter o histórico da conversa.
        """

        self.client = client
        self.system = system
        self.messages = []
        if self.system is not None:
            self.messages.append({"role": "system", "content": self.system})

    def __call__(self, message=""):
        """
        Permite chamar o agente como uma função: agent("pergunta").
        Gerencia o histórico e chama o método execute.
        """

        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        """
        Envia todo o histórico de mensagens para o agente e retorna a resposta.
        """

        completion = self.client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile",
        )
        return completion.choices[0].message.content

## **Prompt do Agente (ReAct)**

Este prompt ensina o modelo a seguir o fluxo: 

*Thought -> Action -> PAUSE -> Observation*

In [39]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

## **Ferramentas**

In [40]:
def calculate(operation):
    """
    Executa uma operação matemática.
    """

    return eval(operation)

In [41]:
def get_planet_mass(planet) -> float:
    """
    Retorna a massa de um planeta.
    """

    match planet.lower():
        case "earth":
            return 5.972e24
        case "mars":
            return 6.39e23
        case "jupiter":
            return 1.898e27
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "neptune":
            return 1.024e26
        case "mercury":
            return 3.285e23
        case "venus":
            return 4.867e24
        case _:
            return 0.0

## **Exemplo de Execução (Manual)**

In [42]:
neil_tyson = Agent(client, system_prompt)

In [43]:
result = neil_tyson("What is the mass of Earth times 5?")
print(result)

Thought: To find the mass of Earth times 5, I first need to determine the mass of Earth. 

Action: get_planet_mass: Earth
PAUSE


In [44]:
print("\n", neil_tyson.messages)


 [{'role': 'system', 'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

In [45]:
observation = get_planet_mass("Earth")
print("\n", observation)


 5.972e+24


In [46]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print("\n", result)


 Thought: Now that I have the mass of Earth, I can multiply it by 5 to find the result.

Action: calculate: 5.972e+24 * 5
PAUSE


In [47]:
print("\n", neil_tyson.messages)


 [{'role': 'system', 'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

In [48]:
observation = calculate("5.972e24 * 5")
print("\n", observation)


 2.9860000000000004e+25


In [49]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print("\n", result)


 Thought: I have now calculated the mass of Earth times 5, so I can provide the answer.

Answer: The mass of Earth times 5 is 2.986e+25.


In [50]:
print("\n", neil_tyson.messages)


 [{'role': 'system', 'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

## **Exemplo de Execução (Automatizado)**

In [51]:
import re

In [52]:
def agent_loop(max_iterations, system, query):
    """
    Automatiza o processo de: 
    1. Perguntar ao agente.
    2. Identificar se ele quer usar uma ferramenta (Action).
    3. Executar a ferramenta e devolver o resultado (Observation).
    4. Repetir até obter o "Answer".
    """

    agent = Agent(client, system)
    tools = ['calculate', 'get_planet_mass']
    next_prompt = query
    i = 0

    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        # Verifica se o modelo quer executar uma ação
        if "PAUSE" in result and "Action" in result:

            # Regex para extrair o nome da ferramenta e o argumento
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                # Executa a função Python
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = f"Observation: Tool {chosen_tool} not found"

            print(next_prompt)

            # Volta para o início do loop com a observação
            continue

        # Para o loop quando o agente tiver a resposta final
        if "Answer" in result:
            break

## **Execução do Agente**

In [53]:
agent_loop(10, system_prompt, "What is the mass of the Earth plus the mass of Mercury, and all of it times 5?")

Thought: To find the mass of the Earth plus the mass of Mercury, and all of it times 5, I first need to find the mass of Earth and Mercury separately. 

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury to add them together. 

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: Now that I have the masses of both Earth and Mercury, I can add them together and then multiply the result by 5.

Action: calculate: (5.972e+24 + 3.285e+23) * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the sum of the masses of Earth and Mercury, and then multiplied it by 5, so I have the final answer.

Answer: The mass of the Earth plus the mass of Mercury, and all of it times 5 is 3.15025e+25.
